In [1]:
import numpy as np
import pandas as pd

df = pd.read_csv("fair_simulation_results_alpha_gamma_ictaly.csv")

# Keep only successful fits for parameter-recovery summaries
if "success" in df.columns:
    df_success = df[df["success"] == True].copy()
else:
    df_success = df.copy()

params = ["mu", "phi", "alpha", "gamma", "r"]

# Empirical bias for each replicate = estimate - truth
for p in params:
    df_success[f"{p}_error"] = (
        df_success[f"{p}_est"] - df_success[f"{p}_true"]
    )

# Define simulation settings
setting_cols = [
    "mu_true",
    "alpha_true",
    "gamma_true",
    "phi_true",
    "r_true"
]

# Empirical bias = mean error across replicates within each setting
bias_summary = (
    df_success
    .groupby(setting_cols)
    .agg(
        n=("gamma_est", "size"),
        mu_bias=("mu_error", "mean"),
        phi_bias=("phi_error", "mean"),
        alpha_bias=("alpha_error", "mean"),
        gamma_bias=("gamma_error", "mean"),
        r_bias=("r_error", "mean"),
    )
    .reset_index()
)

bias_summary

,mu_true,alpha_true,gamma_true,phi_true,r_true,n,mu_bias,phi_bias,alpha_bias,gamma_bias,r_bias
0,-0.15,-5.0,-0.55,0.58,0.048,100,0.005421,-0.501003,2.102598,-0.382633,-0.042540
1,-0.15,-5.0,-0.30,0.58,0.048,100,0.004406,-0.530830,2.064875,-0.242982,-0.042118
2,-0.15,-5.0,1.45,0.58,0.048,100,0.009310,-0.487394,2.081476,-2.050661,-0.042045
3,-0.15,-2.4,-0.55,0.58,0.048,100,0.005887,-0.146942,0.467319,-0.452671,-0.031894
4,-0.15,-2.4,-0.30,0.58,0.048,100,-0.005751,-0.169670,0.376478,-0.225490,-0.032389
5,-0.15,-2.4,1.45,0.58,0.048,100,-0.598584,-0.094516,0.556177,-0.864585,3.411133
6,-0.15,-0.6,-0.55,0.58,0.048,100,-0.003802,-0.019675,0.149724,-0.440857,0.007722
7,-0.15,-0.6,-0.30,0.58,0.048,100,0.013791,-0.033536,0.119856,-0.429704,0.015451
8,-0.15,-0.6,1.45,0.58,0.048,100,-2.172258,-0.059012,1.958843,0.127861,111.944437


In [2]:
import numpy as np
import pandas as pd

params = ["mu", "phi", "alpha", "gamma", "r"]

keys = [
    "mu_true",
    "alpha_true",
    "gamma_true",
    "phi_true",
    "r_true"
]

good = df[df["success"] == True].copy()

rows = []

for setting, g in good.groupby(keys):

    row = dict(zip(keys, setting))
    row["n"] = len(g)

    for p in params:

        true = g[f"{p}_true"].iloc[0]

        # empirical sampling SD across simulation replicates
        sd = g[f"{p}_est"].std(ddof=1)

        lower = g[f"{p}_est"] - 1.96 * sd
        upper = g[f"{p}_est"] + 1.96 * sd

        covered = (
            (true >= lower) &
            (true <= upper)
        )

        row[f"{p}_coverage"] = covered.mean()

    rows.append(row)

coverage = pd.DataFrame(rows)

coverage.round(3)

,mu_true,alpha_true,gamma_true,phi_true,r_true,n,mu_coverage,phi_coverage,alpha_coverage,gamma_coverage,r_coverage
0,-0.15,-5.0,-0.55,0.58,0.048,100,0.94,0.13,0.00,0.94,0.00
1,-0.15,-5.0,-0.30,0.58,0.048,100,0.94,0.06,0.00,0.96,0.00
2,-0.15,-5.0,1.45,0.58,0.048,100,0.96,0.14,0.00,0.78,0.00
3,-0.15,-2.4,-0.55,0.58,0.048,100,0.94,0.82,0.80,0.96,0.00
4,-0.15,-2.4,-0.30,0.58,0.048,100,0.93,0.76,0.87,0.94,0.00
5,-0.15,-2.4,1.45,0.58,0.048,100,0.99,0.92,0.97,0.86,0.97
6,-0.15,-0.6,-0.55,0.58,0.048,100,0.93,0.96,0.96,0.91,0.92
7,-0.15,-0.6,-0.30,0.58,0.048,100,0.93,0.93,0.95,0.90,0.87
8,-0.15,-0.6,1.45,0.58,0.048,100,0.94,0.91,0.97,0.92,0.28


In [3]:
variance_summary = (
    df
    .groupby(setting_cols)
    .agg(
        n=("gamma_est", "size"),

        mu_variance=("mu_est", "var"),
        phi_variance=("phi_est", "var"),
        alpha_variance=("alpha_est", "var"),
        gamma_variance=("gamma_est", "var"),
        r_variance=("r_est", "var"),
    )
    .reset_index()
)

variance_summary

,mu_true,alpha_true,gamma_true,phi_true,r_true,n,mu_variance,phi_variance,alpha_variance,gamma_variance,r_variance
0,-0.15,-5.0,-0.55,0.58,0.048,100,0.001501,0.026673,0.139243,3.982440,0.000001
1,-0.15,-5.0,-0.30,0.58,0.048,100,0.001614,0.022588,0.136372,3.229896,0.000003
2,-0.15,-5.0,1.45,0.58,0.048,100,0.001374,0.025469,0.154685,3.585853,0.000002
3,-0.15,-2.4,-0.55,0.58,0.048,100,0.010389,0.017844,0.203702,1.982551,0.000019
4,-0.15,-2.4,-0.30,0.58,0.048,100,0.012192,0.019437,0.228342,1.964832,0.000017
5,-0.15,-2.4,1.45,0.58,0.048,100,30.448364,0.036509,0.844019,1.142116,411.090183
6,-0.15,-0.6,-0.55,0.58,0.048,100,0.037379,0.014544,0.182259,0.466029,0.000244
7,-0.15,-0.6,-0.30,0.58,0.048,100,0.057838,0.019863,0.239759,0.587785,0.000292
8,-0.15,-0.6,1.45,0.58,0.048,100,1192.709223,0.022266,9.177769,1.865589,2399.913132


In [4]:
import numpy as np
import pandas as pd

df = pd.read_csv("fair_simulation_results_alpha_gamma_post.csv")

# Keep only successful fits for parameter-recovery summaries
if "success" in df.columns:
    df_success = df[df["success"] == True].copy()
else:
    df_success = df.copy()

params = ["mu", "phi", "alpha", "gamma", "r"]

# Empirical bias for each replicate = estimate - truth
for p in params:
    df_success[f"{p}_error"] = (
        df_success[f"{p}_est"] - df_success[f"{p}_true"]
    )

# Define simulation settings
setting_cols = [
    "mu_true",
    "alpha_true",
    "gamma_true",
    "phi_true",
    "r_true"
]

# Empirical bias = mean error across replicates within each setting
bias_summary = (
    df_success
    .groupby(setting_cols)
    .agg(
        n=("gamma_est", "size"),
        mu_bias=("mu_error", "mean"),
        phi_bias=("phi_error", "mean"),
        alpha_bias=("alpha_error", "mean"),
        gamma_bias=("gamma_error", "mean"),
        r_bias=("r_error", "mean"),
    )
    .reset_index()
)

bias_summary

,mu_true,alpha_true,gamma_true,phi_true,r_true,n,mu_bias,phi_bias,alpha_bias,gamma_bias,r_bias
0,-0.16,-5.0,-3.00,0.21,0.0009,100,0.001368,-0.039552,-0.027187,-0.079213,-0.000261
1,-0.16,-5.0,-0.30,0.21,0.0009,100,0.001390,-0.045979,-0.016430,-0.694967,-0.000159
2,-0.16,-5.0,1.45,0.21,0.0009,100,-0.000890,-0.011552,0.025341,-1.777616,-0.000052
3,-0.16,-2.4,-3.00,0.21,0.0009,100,-0.004475,-0.016823,-0.001249,-1.204175,0.004164
4,-0.16,-2.4,-0.30,0.21,0.0009,100,0.007703,-0.026518,-0.048330,-0.718875,0.007694
5,-0.16,-2.4,1.45,0.21,0.0009,100,0.004079,-0.020916,-0.145230,-0.059936,0.013917
6,-0.16,-0.6,-3.00,0.21,0.0009,100,0.003690,0.012600,0.052841,-1.051668,0.014627
7,-0.16,-0.6,-0.30,0.21,0.0009,100,-0.008251,-0.012039,0.039905,-0.465121,0.047622
8,-0.16,-0.6,1.45,0.21,0.0009,100,1.093953,-0.032677,1.499336,0.016469,101.870556


In [5]:
import numpy as np
import pandas as pd

params = ["mu", "phi", "alpha", "gamma", "r"]

keys = [
    "mu_true",
    "alpha_true",
    "gamma_true",
    "phi_true",
    "r_true"
]

good = df[df["success"] == True].copy()

rows = []

for setting, g in good.groupby(keys):

    row = dict(zip(keys, setting))
    row["n"] = len(g)

    for p in params:

        true = g[f"{p}_true"].iloc[0]

        # empirical sampling SD across simulation replicates
        sd = g[f"{p}_est"].std(ddof=1)

        lower = g[f"{p}_est"] - 1.96 * sd
        upper = g[f"{p}_est"] + 1.96 * sd

        covered = (
            (true >= lower) &
            (true <= upper)
        )

        row[f"{p}_coverage"] = covered.mean()

    rows.append(row)

coverage = pd.DataFrame(rows)

coverage.round(3)

,mu_true,alpha_true,gamma_true,phi_true,r_true,n,mu_coverage,phi_coverage,alpha_coverage,gamma_coverage,r_coverage
0,-0.16,-5.0,-3.00,0.21,0.001,100,0.95,0.93,0.94,0.91,0.55
1,-0.16,-5.0,-0.30,0.21,0.001,100,0.94,0.93,0.95,1.00,0.89
2,-0.16,-5.0,1.45,0.21,0.001,100,0.96,0.95,0.96,1.00,0.95
3,-0.16,-2.4,-3.00,0.21,0.001,100,0.94,0.95,0.94,0.98,0.05
4,-0.16,-2.4,-0.30,0.21,0.001,100,0.95,0.95,0.94,0.92,0.02
5,-0.16,-2.4,1.45,0.21,0.001,100,0.95,0.95,0.94,0.94,0.09
6,-0.16,-0.6,-3.00,0.21,0.001,100,0.94,0.95,0.96,0.58,0.00
7,-0.16,-0.6,-0.30,0.21,0.001,100,0.97,0.95,0.95,0.92,0.16
8,-0.16,-0.6,1.45,0.21,0.001,100,0.91,0.96,0.82,0.92,0.39


In [4]:
variance_summary = (
    df
    .groupby(setting_cols)
    .agg(
        n=("gamma_est", "size"),

        mu_variance=("mu_est", "var"),
        phi_variance=("phi_est", "var"),
        alpha_variance=("alpha_est", "var"),
        gamma_variance=("gamma_est", "var"),
        r_variance=("r_est", "var"),
    )
    .reset_index()
)

variance_summary

,mu_true,alpha_true,gamma_true,phi_true,r_true,n,mu_variance,phi_variance,alpha_variance,gamma_variance,r_variance
0,-0.16,-5.0,-3.00,0.21,0.0009,100,0.000137,0.017167,0.072356,8.382983,2.305489e-08
1,-0.16,-5.0,-0.30,0.21,0.0009,100,0.000193,0.020961,0.102160,13.681197,4.252290e-08
2,-0.16,-5.0,1.45,0.21,0.0009,100,0.000239,0.022081,0.109570,14.445267,6.200106e-08
3,-0.16,-2.4,-3.00,0.21,0.0009,100,0.000914,0.008924,0.074595,1.170593,1.425656e-06
4,-0.16,-2.4,-0.30,0.21,0.0009,100,0.002000,0.028492,0.135102,2.244198,3.958577e-06
5,-0.16,-2.4,1.45,0.21,0.0009,100,0.002833,0.021133,0.127700,0.983442,2.168967e-05
6,-0.16,-0.6,-3.00,0.21,0.0009,100,0.001994,0.003388,0.083204,0.401668,1.111169e-05
7,-0.16,-0.6,-0.30,0.21,0.0009,100,0.015529,0.018433,0.181670,0.602705,3.060650e-04
8,-0.16,-0.6,1.45,0.21,0.0009,100,410.332848,0.025778,5.885143,1.447458,2.837204e+03


In [6]:
import numpy as np
import pandas as pd

df = pd.read_csv("fair_simulation_results_alpha_gamma_pre.csv")

# Keep only successful fits for parameter-recovery summaries
if "success" in df.columns:
    df_success = df[df["success"] == True].copy()
else:
    df_success = df.copy()

params = ["mu", "phi", "alpha", "gamma", "r"]

# Empirical bias for each replicate = estimate - truth
for p in params:
    df_success[f"{p}_error"] = (
        df_success[f"{p}_est"] - df_success[f"{p}_true"]
    )

# Define simulation settings
setting_cols = [
    "mu_true",
    "alpha_true",
    "gamma_true",
    "phi_true",
    "r_true"
]

# Empirical bias = mean error across replicates within each setting
bias_summary = (
    df_success
    .groupby(setting_cols)
    .agg(
        n=("gamma_est", "size"),
        mu_bias=("mu_error", "mean"),
        phi_bias=("phi_error", "mean"),
        alpha_bias=("alpha_error", "mean"),
        gamma_bias=("gamma_error", "mean"),
        r_bias=("r_error", "mean"),
    )
    .reset_index()
)

bias_summary

,mu_true,alpha_true,gamma_true,phi_true,r_true,n,mu_bias,phi_bias,alpha_bias,gamma_bias,r_bias
0,-0.15,-7.98,-3.50,0.22,0.001,100,-0.001018,-0.165192,1.225389,2.308249,-0.000866
1,-0.15,-7.98,-0.03,0.22,0.001,100,-0.000320,-0.155841,1.220636,-1.232470,-0.000870
2,-0.15,-7.98,1.45,0.22,0.001,100,-0.000748,-0.179371,1.229416,-2.819554,-0.000867
3,-0.15,-5.08,-3.50,0.22,0.001,100,0.001607,-0.056054,-0.041407,0.766906,-0.000395
4,-0.15,-5.08,-0.03,0.22,0.001,100,0.000973,-0.043016,0.080630,-1.488751,-0.000268
5,-0.15,-5.08,1.45,0.22,0.001,100,-0.000374,-0.028692,0.106211,-2.110082,-0.000199
6,-0.15,-2.52,-3.50,0.22,0.001,100,-0.000099,-0.011360,-0.151399,-0.414959,0.003204
7,-0.15,-2.52,-0.03,0.22,0.001,100,0.007183,-0.022724,-0.059908,-0.731425,0.007201
8,-0.15,-2.52,1.45,0.22,0.001,100,0.007455,-0.027942,-0.111020,-0.308620,0.011886


In [8]:
import numpy as np
import pandas as pd

params = ["mu", "phi", "alpha", "gamma", "r"]

keys = [
    "mu_true",
    "alpha_true",
    "gamma_true",
    "phi_true",
    "r_true"
]

good = df[df["success"] == True].copy()

rows = []

for setting, g in good.groupby(keys):

    row = dict(zip(keys, setting))
    row["n"] = len(g)

    for p in params:

        true = g[f"{p}_true"].iloc[0]

        # empirical sampling SD across simulation replicates
        sd = g[f"{p}_est"].std(ddof=1)

        lower = g[f"{p}_est"] - 1.96 * sd
        upper = g[f"{p}_est"] + 1.96 * sd

        covered = (
            (true >= lower) &
            (true <= upper)
        )

        row[f"{p}_coverage"] = covered.mean()

    rows.append(row)

coverage = pd.DataFrame(rows)

coverage.round(3)

,mu_true,alpha_true,gamma_true,phi_true,r_true,n,mu_coverage,phi_coverage,alpha_coverage,gamma_coverage,r_coverage
0,-0.15,-7.98,-3.50,0.22,0.001,100,0.96,0.74,0.00,0.77,0.00
1,-0.15,-7.98,-0.03,0.22,0.001,100,0.95,0.78,0.00,1.00,0.00
2,-0.15,-7.98,1.45,0.22,0.001,100,0.94,0.74,0.01,1.00,0.00
3,-0.15,-5.08,-3.50,0.22,0.001,100,0.95,0.94,0.91,0.91,0.19
4,-0.15,-5.08,-0.03,0.22,0.001,100,0.96,0.96,0.96,1.00,0.71
5,-0.15,-5.08,1.45,0.22,0.001,100,0.97,0.92,0.94,1.00,0.87
6,-0.15,-2.52,-3.50,0.22,0.001,100,0.93,0.94,0.94,0.96,0.02
7,-0.15,-2.52,-0.03,0.22,0.001,100,0.95,0.94,0.95,0.90,0.03
8,-0.15,-2.52,1.45,0.22,0.001,100,0.96,0.93,0.94,0.92,0.34


In [6]:
variance_summary = (
    df
    .groupby(setting_cols)
    .agg(
        n=("gamma_est", "size"),

        mu_variance=("mu_est", "var"),
        phi_variance=("phi_est", "var"),
        alpha_variance=("alpha_est", "var"),
        gamma_variance=("gamma_est", "var"),
        r_variance=("r_est", "var"),
    )
    .reset_index()
)

variance_summary

,mu_true,alpha_true,gamma_true,phi_true,r_true,n,mu_variance,phi_variance,alpha_variance,gamma_variance,r_variance
0,-0.15,-7.98,-3.50,0.22,0.001,100,0.000023,0.017620,0.047177,18.242944,1.247420e-09
1,-0.15,-7.98,-0.03,0.22,0.001,100,0.000025,0.019146,0.038176,17.871739,9.666967e-10
2,-0.15,-7.98,1.45,0.22,0.001,100,0.000024,0.019296,0.054442,17.837458,1.594816e-09
3,-0.15,-5.08,-3.50,0.22,0.001,100,0.000189,0.017763,0.074510,9.809973,2.275313e-08
4,-0.15,-5.08,-0.03,0.22,0.001,100,0.000178,0.019843,0.079324,11.705649,3.462799e-08
5,-0.15,-5.08,1.45,0.22,0.001,100,0.000272,0.021530,0.095921,14.626437,3.584271e-08
6,-0.15,-2.52,-3.50,0.22,0.001,100,0.001073,0.012267,0.094335,2.136657,7.101380e-07
7,-0.15,-2.52,-0.03,0.22,0.001,100,0.001841,0.025550,0.169431,3.848979,3.849867e-06
8,-0.15,-2.52,1.45,0.22,0.001,100,0.003790,0.025746,0.128748,1.815871,2.328170e-05
